# Markdown to XML

The purpose of this notebook is to convert my Markdown transcription of Giarratano's apparatus criticus to the XML snippet that I've settled on for the transcription format in the JSON files. I will also make the `apparatus_entry` JSON object while I'm at it. 

In [56]:
import pandas as pd
import json
import re
from datetime import datetime, timezone

I manually transcribed the apparatus criticus entries for lines 14–end of Calp. Sic. 1 in Giarratano's 1910 edition. 1–13 were already done before I decided to accelerate the process with some automation. I used Markdown because it was easier for me to type than all the angle brackets for XML.

In [57]:
df = pd.read_csv('../converted-json/entries.csv')

In [58]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 204 entries, 0 to 203
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   line_number  204 non-null    int64
 1   lemma        198 non-null    str  
 2   entry        204 non-null    str  
 3   note         13 non-null     str  
dtypes: int64(1), str(3)
memory usage: 6.5 KB


In [59]:
df

,line_number,lemma,entry,note
0,14,nocturnaque,"*noturnaque* G,*no^c^turnaque* N (*no^c^turna*...",NaN
1,15,pervia,*previa* P.,NaN
2,15,Fauni,"*fanni* P.,\n16,None,*O*. φεγην edd. praeter B...",NaN
3,16,calamos,*chalamos* G. versum om. b; post hunc versum B...,NaN
4,17,mea,"*mea* om. ρ (add. A in mg.), *nova* νφηθrson B...",NaN
...,...,...,...,...
199,93,sonum modulemur avena,*sono modulemur avenae* Wernsd.,NaN
200,94,forsitan,*forsitā* N.,NaN
201,94,augustas,"*angustas* Nπφρ^1^λ^1^, *agustas* μ.",NaN
202,94,Meliboeus,"*meliboeus* νε^2^ρ^2^, *melibeus* GPλπδ^1^γρ^1...",NaN


Here's a function to convert the Markdown to XML.

In [98]:
def markdown_to_xml(text):
    italics = re.sub(r'\*(.*?)\*', r'<hi rend="italics">\1</hi>', text)
    superscript = re.sub(r'\^(.(.*?))\^', r'<hi rend="superscript">\1</hi>', italics)
    xml = "<ab>" + superscript + "</ab>"
    return xml

I'll make a dictionary of the Markdown transcription and XML.

In [99]:
df["xml"] = df["entry"].apply(markdown_to_xml)

In [100]:
df

,line_number,lemma,entry,note,xml,cts_id,validate
0,14,nocturnaque,"*noturnaque* G,*no^c^turnaque* N (*no^c^turna*...",NaN,"<ab><hi rend=""italics"">noturnaque</hi> G,<hi r...",urn:cts:latinLit:phi0830.phi001.dll:1.14@noctu...,True
1,15,pervia,*previa* P.,NaN,"<ab><hi rend=""italics"">previa</hi> P.</ab>",urn:cts:latinLit:phi0830.phi001.dll:1.15@pervia,True
2,15,Fauni,"*fanni* P.,\n16,None,*O*. φεγην edd. praeter B...",NaN,"<ab><hi rend=""italics"">fanni</hi> P.,\n16,None...",urn:cts:latinLit:phi0830.phi001.dll:1.15@Fauni,True
3,16,calamos,*chalamos* G. versum om. b; post hunc versum B...,NaN,"<ab><hi rend=""italics"">chalamos</hi> G. versum...",urn:cts:latinLit:phi0830.phi001.dll:1.16@calamos,True
4,17,mea,"*mea* om. ρ (add. A in mg.), *nova* νφηθrson B...",NaN,"<ab><hi rend=""italics"">mea</hi> om. ρ (add. A ...",urn:cts:latinLit:phi0830.phi001.dll:1.17@mea,True
...,...,...,...,...,...,...,...
199,93,sonum modulemur avena,*sono modulemur avenae* Wernsd.,NaN,"<ab><hi rend=""italics"">sono modulemur avenae</...",urn:cts:latinLit:phi0830.phi001.dll:1.93@sonum...,True
200,94,forsitan,*forsitā* N.,NaN,"<ab><hi rend=""italics"">forsitā</hi> N.</ab>",urn:cts:latinLit:phi0830.phi001.dll:1.94@forsitan,True
201,94,augustas,"*angustas* Nπφρ^1^λ^1^, *agustas* μ.",NaN,"<ab><hi rend=""italics"">angustas</hi> Nπφρ<hi r...",urn:cts:latinLit:phi0830.phi001.dll:1.94@augustas,True
202,94,Meliboeus,"*meliboeus* νε^2^ρ^2^, *melibeus* GPλπδ^1^γρ^1...",NaN,"<ab><hi rend=""italics"">meliboeus</hi> νε<hi re...",urn:cts:latinLit:phi0830.phi001.dll:1.94@Melib...,True


In [101]:
def create_cts_id(row):
    if row['lemma'] is None or pd.isna(row['lemma']):
        return f"urn:cts:latinLit:phi0830.phi001.dll:1.{row['line_number']}@"
    else:
        return f"urn:cts:latinLit:phi0830.phi001.dll:1.{row['line_number']}@{row['lemma']}"

df["cts_id"] = df.apply(create_cts_id, axis=1)

In [102]:
df["cts_id"]

0      urn:cts:latinLit:phi0830.phi001.dll:1.14@noctu...
1        urn:cts:latinLit:phi0830.phi001.dll:1.15@pervia
2         urn:cts:latinLit:phi0830.phi001.dll:1.15@Fauni
3       urn:cts:latinLit:phi0830.phi001.dll:1.16@calamos
4           urn:cts:latinLit:phi0830.phi001.dll:1.17@mea
                             ...                        
199    urn:cts:latinLit:phi0830.phi001.dll:1.93@sonum...
200    urn:cts:latinLit:phi0830.phi001.dll:1.94@forsitan
201    urn:cts:latinLit:phi0830.phi001.dll:1.94@augustas
202    urn:cts:latinLit:phi0830.phi001.dll:1.94@Melib...
203       urn:cts:latinLit:phi0830.phi001.dll:1.94@aures
Name: cts_id, Length: 204, dtype: str

In [103]:
id_cts = {}

# Read json file
with open('../converted-json/calpSicPoem1-target-array.json', 'r') as f:
    data = json.load(f)
    # Loop through the array of objects and create a dictionary with id as key and cts as value
    for item in data:
        # If the apparatus_entry_id is empty, skip this item
        if "apparatus_entry_id" not in item or not item["apparatus_entry_id"]:
            continue
        else:
            id_cts[item['apparatus_entry_id']] = item['target']

In [104]:
id_cts

{'urn:uuid:f2e93fa5-f815-4ff4-92ca-adaa365f8f20': ['urn:cts:latinLit:phi0830.phi001.dll:1.1@C.'],
 'urn:uuid:37b666e6-9980-4776-81d9-d056936f49c5': ['urn:cts:latinLit:phi0830.phi001.dll:1.1@Nondum'],
 'urn:uuid:fc07310e-e32e-42f5-b6b8-7915871f71bc': ['urn:cts:latinLit:phi0830.phi001.dll:1.1@declinis'],
 'urn:uuid:c8f6d0a9-7c57-4a5e-9def-3ca62cf2e188': ['urn:cts:latinLit:phi0830.phi001.dll:1.2@quamvis'],
 'urn:uuid:7757d8c7-4980-42be-bf21-28ccf1a2659f': ['urn:cts:latinLit:phi0830.phi001.dll:1.2@praela'],
 'urn:uuid:022529cf-6b85-48c8-8dee-6b53245e3c44': ['urn:cts:latinLit:phi0830.phi001.dll:1.3'],
 'urn:uuid:4817a744-94b3-4f6c-a2ec-d57ca52daef9': ['urn:cts:latinLit:phi0830.phi001.dll:1.3@musta'],
 'urn:uuid:90c8a150-48bf-43e3-b9b2-8ae8d02b2d6f': ['urn:cts:latinLit:phi0830.phi001.dll:1.4@'],
 'urn:uuid:3f8dc9f9-4f40-4967-b101-191250643d35': ['urn:cts:latinLit:phi0830.phi001.dll:1.4@Ornyte'],
 'urn:uuid:882ea6a9-53ad-4616-a73d-7831b4cdaa8a': ['urn:cts:latinLit:phi0830.phi001.dll:1.4@vacca

In [105]:
cts = [" ".join(v) for k, v in id_cts.items()]

In [106]:
df["validate"] = df["cts_id"].apply(lambda x: x in cts)

In [107]:
df["validate"].value_counts()

validate
True     194
False     10
Name: count, dtype: int64

In [108]:
false_entries = df[df["validate"] == False].copy()

In [109]:
for id in false_entries["cts_id"]:
    print(id)

urn:cts:latinLit:phi0830.phi001.dll:1.23@se
urn:cts:latinLit:phi0830.phi001.dll:1.30@nec montana sacros distinguunt iubila versus.
urn:cts:latinLit:phi0830.phi001.dll:1.31@
urn:cts:latinLit:phi0830.phi001.dll:1.31@C.
urn:cts:latinLit:phi0830.phi001.dll:1.31@oculoque
urn:cts:latinLit:phi0830.phi001.dll:1.31@sequaci
urn:cts:latinLit:phi0830.phi001.dll:1.32@perlege
urn:cts:latinLit:phi0830.phi001.dll:1.57@
urn:cts:latinLit:phi0830.phi001.dll:1.68@inter-inter
urn:cts:latinLit:phi0830.phi001.dll:1.87@nec prius ex meritis defunctos Roma penates


In [110]:
df.to_csv('../converted-json/entries_with_cts5.csv', index=False)